In [0]:
import pyspark.sql.functions as F
from pyspark.sql.utils import StreamingQueryException
from pathlib import Path
import json

config_route = Path("/Workspace/Users/lareddy.abhi@gmail.com/f1-data-lakehouse/config/adb-config-f1.json")
config_json = config_route.read_text(encoding="utf-8")
config_dict = json.loads(config_json)

config_dict["openf1_endpoints"].append("sessions")

all_endpoints = config_dict["openf1_endpoints"] + config_dict["openf1_telemetry_ends"]
  
def write_data_to_delta(endpoint):
    try:
        df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format","json")
            .option("cloudFiles.schemaLocation",f"/Volumes/f1_lakehouse_dev/ops/checkpoints/bronze/{endpoint}")
            .option("multiLine","true")
            .option("rescuedDataColumn", f"_rescued_data")
            .load(f"/Volumes/f1_lakehouse_dev/landing/raw/openf1/{endpoint}")    
        )
        print(f"reading data from source {endpoint}")
        df_transformed = (
            df
            .withColumn("_source_file_path", F.col("_metadata.file_path"))
            .withColumn("_modification_time", F.col("_metadata.file_modification_time"))
            .withColumn("_ingestion_time", F.current_timestamp())
        )

        query = (
            df_transformed.writeStream
            .format("delta")
            .outputMode("append") 
            .option("checkpointLocation", f"/Volumes/f1_lakehouse_dev/ops/checkpoints/bronze/{endpoint}")
            .trigger(availableNow=True)
            .toTable(f"f1_lakehouse_dev.bronze.{endpoint}")
        )
        query.awaitTermination()
        print(f"Succefully wrote {endpoint}")
    except StreamingQueryException:
        raise ValueError(f"Exited with error: {StreamingQueryException.getMessage}")


# for epoint in all_endpoints:
#     write_data_to_delta(epoint)
write_data_to_delta("laps")